# Model Comparison for First-Phase Structural Prediction

This notebook compares seven regressors for `Grain_1`, `a_1`, `b_1`,
and `c_1` using iteration 6 datasets: XGBoost, Ridge, Gaussian Process,
Kernel Ridge, Random Forest, Gradient Boosting, and TabPFN.

## Workflow

1. Aggregate records by compound, retaining the first feature values
   and averaging target values.
2. Split each target dataset into training and test sets (80/20).
3. Select features using XGBoost: 20 for Grain_1 and 15 for each
   lattice parameter.
4. Evaluate model configurations using five-fold cross-validation
   with MAE as the selection metric.
5. Export comparison metrics ranked by test R².

Grain_1 training, evaluation, and exported predictions use the log1p
scale. Lattice parameters use their original scales.
Feature selection occurs before cross-validation.
TabPFN runs on CPU using the specified local checkpoint.

Each target section can run independently. Complete its cells in order,
including the comparison export, before switching targets.

## grain1

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_Grain1.csv"

TARGET_COL = "Grain_1"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 20
TEST_SIZE = 0.20
RANDOM_STATE = 42

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/Grain1"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train_original = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test_original = (
    test_data[TARGET_COL]
    .to_numpy()
)

y_train_log = np.log1p(
    y_train_original
)

y_test_log = np.log1p(
    y_test_original
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 156
Training samples: 124
Test samples: 32
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train_log
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")


Selected features:
1. C_D
2. C_AW
3. AB_Nv
4. AB_AR
5. AB_AN
6. AB_Nd
7. AB_CR
8. C_FIE
9. AB_EC
10. AB_FIE
11. C_CR
12. AB_D
13. AB_EA
14. C_AN
15. AB_SIE
16. AB_EN
17. AB_AW
18. C_AR
19. AB_TC
20. C_SIE


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train_log
    )

    best_model = search.best_estimator_

    train_prediction_log = best_model.predict(
        X_train
    )

    test_prediction_log = best_model.predict(
        X_test
    )

    train_r2_log = r2_score(
        y_train_log,
        train_prediction_log
    )

    train_mae_log = mean_absolute_error(
        y_train_log,
        train_prediction_log
    )

    train_rmse_log = np.sqrt(
        mean_squared_error(
            y_train_log,
            train_prediction_log
        )
    )

    test_r2_log = r2_score(
        y_test_log,
        test_prediction_log
    )

    test_mae_log = mean_absolute_error(
        y_test_log,
        test_prediction_log
    )

    test_rmse_log = np.sqrt(
        mean_squared_error(
            y_test_log,
            test_prediction_log
        )
    )

    print(f"\n===== {model_name} =====")

    print(
        "Best parameters:",
        search.best_params_
    )

    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )

    print(
        f"Train: R2={train_r2_log:.6f} | "
        f"MAE={train_mae_log:.6f} | "
        f"RMSE={train_rmse_log:.6f}"
    )

    print(
        f"Test: R2={test_r2_log:.6f} | "
        f"MAE={test_mae_log:.6f} | "
        f"RMSE={test_rmse_log:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE_log": -search.best_score_,
        "Train_R2_log": train_r2_log,
        "Train_MAE_log": train_mae_log,
        "Train_RMSE_log": train_rmse_log,
        "Test_R2_log": test_r2_log,
        "Test_MAE_log": test_mae_log,
        "Test_RMSE_log": test_rmse_log,
        "Best_parameters": str(
            search.best_params_
        )
    }

    prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true_log": y_test_log,
        "y_pred_log": test_prediction_log
    })

    prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
#     min_child_weight=4,
#     gamma=0.8,
#     subsample=0.8,
#     colsample_bytree=0.7,
#     reg_alpha=0.1,
#     reg_lambda=8,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200,
        300,
        500,
        700
    ],
    "max_depth": [
        1,
        2,
        3,
        6,
        8
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
        0.20,
        0.40
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)



===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300}
Five-fold CV MAE: 0.298212
Train: R2=0.999995 | MAE=0.001212 | RMSE=0.002013
Test: R2=0.841108 | MAE=0.252417 | RMSE=0.362035


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 0.1}
Five-fold CV MAE: 0.532454
Train: R2=0.520432 | MAE=0.461233 | RMSE=0.602334
Test: R2=0.485213 | MAE=0.533401 | RMSE=0.651648


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 0.01}
Five-fold CV MAE: 0.253882
Train: R2=0.979269 | MAE=0.083903 | RMSE=0.125235
Test: R2=0.804845 | MAE=0.302238 | RMSE=0.401226


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.1}
Five-fold CV MAE: 0.300799
Train: R2=0.927287 | MAE=0.168040 | RMSE=0.234541
Test: R2=0.464536 | MAE=0.369816 | RMSE=0.664606


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.333967
Train: R2=0.968062 | MAE=0.105716 | RMSE=0.155440
Test: R2=0.773412 | MAE=0.304533 | RMSE=0.432332


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 200}
Five-fold CV MAE: 0.300173
Train: R2=0.996518 | MAE=0.030731 | RMSE=0.051322
Test: R2=0.803507 | MAE=0.307668 | RMSE=0.402599


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)


===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.212690
Train: R2=0.972776 | MAE=0.111541 | RMSE=0.143512
Test: R2=0.809099 | MAE=0.266813 | RMSE=0.396829


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    krr_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2_log",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE_log",
            "Test_R2_log",
            "Test_MAE_log",
            "Test_RMSE_log"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results_log.csv",
    index=False
)


Final model comparison:
               Model  CV_MAE_log  Test_R2_log  Test_MAE_log  Test_RMSE_log
0            XGBoost    0.298212     0.841108      0.252417       0.362035
1             TabPFN    0.212690     0.809099      0.266813       0.396829
2                GPR    0.253882     0.804845      0.302238       0.401226
3  Gradient Boosting    0.300173     0.803507      0.307668       0.402599
4      Random Forest    0.333967     0.773412      0.304533       0.432332
5              Ridge    0.532454     0.485213      0.533401       0.651648
6       Kernel Ridge    0.300799     0.464536      0.369816       0.664606


## a1

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_a1.csv"

TARGET_COL = "a_1"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 15
TEST_SIZE = 0.20
RANDOM_STATE = 2027

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/a1"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 156
Training samples: 124
Test samples: 32
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")


Selected features:
1. C_CR
2. C_FIE
3. AB_CR
4. AB_AR
5. AB_EC
6. AB_AW
7. AB_EN
8. AB_SIE
9. AB_Nv
10. C_D
11. C_AN
12. AB_TC
13. AB_EA
14. AB_FIE
15. AB_D


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200,
        300,
        400,
        500,
        700
    ],
    "max_depth": [
        3,
        6,
        8,
        9,
        10
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05,
        0.10,
        0.20,
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.03, 'max_depth': 9, 'n_estimators': 400}
Five-fold CV MAE: 0.349052
Train: R2=0.999999 | MAE=0.000881 | RMSE=0.001980
Test: R2=0.958786 | MAE=0.087353 | RMSE=0.296438


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)


===== Ridge =====
Best parameters: {'model__alpha': 0.1}
Five-fold CV MAE: 0.981069
Train: R2=0.620276 | MAE=0.865260 | RMSE=1.245559
Test: R2=0.327467 | MAE=0.911291 | RMSE=1.197484


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-06}
Five-fold CV MAE: 0.425007
Train: R2=0.998153 | MAE=0.031720 | RMSE=0.086865
Test: R2=0.809030 | MAE=0.322273 | RMSE=0.638109


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)


===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.001, 'model__gamma': 0.1}
Five-fold CV MAE: 0.417592
Train: R2=0.958213 | MAE=0.159163 | RMSE=0.413189
Test: R2=0.989385 | MAE=0.105327 | RMSE=0.150441


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.458451
Train: R2=0.964191 | MAE=0.144658 | RMSE=0.382496
Test: R2=0.936315 | MAE=0.162101 | RMSE=0.368495


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 200}
Five-fold CV MAE: 0.385958
Train: R2=0.999932 | MAE=0.011455 | RMSE=0.016674
Test: R2=0.954815 | MAE=0.151553 | RMSE=0.310392


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.223392
Train: R2=0.995770 | MAE=0.051689 | RMSE=0.131469
Test: R2=0.995822 | MAE=0.048271 | RMSE=0.094381


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    krr_result,

])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0             TabPFN  0.223392  0.995822  0.048271   0.094381
1       Kernel Ridge  0.417592  0.989385  0.105327   0.150441
2            XGBoost  0.349052  0.958786  0.087353   0.296438
3  Gradient Boosting  0.385958  0.954815  0.151553   0.310392
4      Random Forest  0.458451  0.936315  0.162101   0.368495
5                GPR  0.425007  0.809030  0.322273   0.638109
6              Ridge  0.981069  0.327467  0.911291   1.197484


## b1

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)


D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_b1.csv"

TARGET_COL = "b_1"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 15
TEST_SIZE = 0.20
RANDOM_STATE = 2027

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/b1"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 156
Training samples: 124
Test samples: 32
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")



Selected features:
1. C_VR
2. AB_CR
3. C_FIE
4. AB_AW
5. AB_AR
6. AB_EN
7. AB_EC
8. AB_AN
9. AB_EA
10. AB_TC
11. AB_FIE
12. C_AW
13. C_CR
14. AB_Nd
15. C_D


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Five-fold CV MAE: 0.441122
Train: R2=0.999767 | MAE=0.019952 | RMSE=0.031880
Test: R2=0.969517 | MAE=0.112344 | RMSE=0.254941


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 0.1}
Five-fold CV MAE: 1.076962
Train: R2=0.482894 | MAE=0.955781 | RMSE=1.501533
Test: R2=0.494489 | MAE=0.831168 | RMSE=1.038193


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-08}
Five-fold CV MAE: 0.421276
Train: R2=0.998793 | MAE=0.025523 | RMSE=0.072543
Test: R2=0.813343 | MAE=0.320458 | RMSE=0.630862


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
    ],
    "model__gamma": [
        0.001,
        0.01,
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.01}
Five-fold CV MAE: 0.784802
Train: R2=0.741740 | MAE=0.595929 | RMSE=1.061141
Test: R2=0.852468 | MAE=0.454195 | RMSE=0.560861


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.508272
Train: R2=0.938607 | MAE=0.176435 | RMSE=0.517372
Test: R2=0.931244 | MAE=0.170010 | RMSE=0.382884


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 200}
Five-fold CV MAE: 0.438199
Train: R2=0.999914 | MAE=0.012282 | RMSE=0.019365
Test: R2=0.942285 | MAE=0.166313 | RMSE=0.350799


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.303695
Train: R2=0.983646 | MAE=0.096788 | RMSE=0.267025
Test: R2=0.995951 | MAE=0.048709 | RMSE=0.092912


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    gpr_result,
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0             TabPFN  0.303695  0.995951  0.048709   0.092912
1            XGBoost  0.441122  0.969517  0.112344   0.254941
2  Gradient Boosting  0.438199  0.942285  0.166313   0.350799
3      Random Forest  0.508272  0.931244  0.170010   0.382884
4       Kernel Ridge  0.784802  0.852468  0.454195   0.560861
5                GPR  0.421276  0.813343  0.320458   0.630862
6              Ridge  1.076962  0.494489  0.831168   1.038193


## c1

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)


D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_c1.csv"

TARGET_COL = "c_1"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 15
TEST_SIZE = 0.20
RANDOM_STATE = 2027

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/c1"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 156
Training samples: 124
Test samples: 32
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")



Selected features:
1. C_AN
2. AB_FIE
3. C_AW
4. AB_EC
5. AB_TC
6. C_D
7. AB_AR
8. AB_AN
9. AB_D
10. AB_EN
11. AB_EA
12. AB_SIE
13. AB_CR
14. AB_Nd
15. AB_AW


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Five-fold CV MAE: 0.257336
Train: R2=0.999940 | MAE=0.012541 | RMSE=0.020127
Test: R2=0.998618 | MAE=0.075867 | RMSE=0.115372


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 0.01}
Five-fold CV MAE: 1.315122
Train: R2=0.632317 | MAE=1.067275 | RMSE=1.570247
Test: R2=0.674209 | MAE=1.366386 | RMSE=1.771531


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 0.0001}
Five-fold CV MAE: 0.210104
Train: R2=0.998488 | MAE=0.043912 | RMSE=0.100698
Test: R2=0.990310 | MAE=0.167442 | RMSE=0.305524


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
    ],
    "model__gamma": [
        0.001,
        0.01,
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.01}
Five-fold CV MAE: 0.481731
Train: R2=0.962713 | MAE=0.354830 | RMSE=0.500048
Test: R2=0.968789 | MAE=0.358273 | RMSE=0.548314


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.335931
Train: R2=0.981445 | MAE=0.113232 | RMSE=0.352745
Test: R2=0.998135 | MAE=0.078106 | RMSE=0.134028


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'min_samples_leaf': 1, 'n_estimators': 200}
Five-fold CV MAE: 0.270844
Train: R2=0.999327 | MAE=0.038176 | RMSE=0.067170
Test: R2=0.995497 | MAE=0.133946 | RMSE=0.208279


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.219331
Train: R2=0.999390 | MAE=0.022452 | RMSE=0.063955
Test: R2=0.998380 | MAE=0.067825 | RMSE=0.124934


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    gpr_result,
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0            XGBoost  0.257336  0.998618  0.075867   0.115372
1             TabPFN  0.219331  0.998380  0.067825   0.124934
2      Random Forest  0.335931  0.998135  0.078106   0.134028
3  Gradient Boosting  0.270844  0.995497  0.133946   0.208279
4                GPR  0.210104  0.990310  0.167442   0.305524
5       Kernel Ridge  0.481731  0.968789  0.358273   0.548314
6              Ridge  1.315122  0.674209  1.366386   1.771531


In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "tabpfn",
    "ngboost",
    "torch"
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

try:
    import torch
    print("CUDA:", torch.version.cuda)
except ImportError:
    print("CUDA: unavailable")

Python: 3.11.15
numpy: 2.4.6
pandas: 2.3.3
scikit-learn: 1.9.0
xgboost: not installed
tabpfn: 8.3.0
ngboost: 0.5.11
torch: 2.13.0+cpu
CUDA: None
